In [ ]:
# Importe
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


In [ ]:
# Pfade und Eingabedateien
PROJECT_ROOT = Path('..').resolve()
PRED_ROOT_CANDIDATES = [PROJECT_ROOT / 'outputs' / 'comparative_prediction_final_robustness']
RECON_CORE_CANDIDATES = [PROJECT_ROOT / 'outputs' / 'benchmark_reconciliation_two_perspectives' / 'tables' / '19_case_level_reconciliation_core.csv', PROJECT_ROOT / 'outputs' / 'benchmark_reconciliation_two_perspectives' / '19_case_level_reconciliation_core.csv']
PRED_ROOT = next((p for p in PRED_ROOT_CANDIDATES if (p / 'tables' / '13_selected_models_test_results.csv').exists()), None)
if PRED_ROOT is None:
    PRED_ROOT = next((p for p in PRED_ROOT_CANDIDATES if (p / '13_selected_models_test_results.csv').exists()), None)
RECON_CORE_PATH = next((p for p in RECON_CORE_CANDIDATES if p.exists()), None)
if PRED_ROOT is None:
    raise FileNotFoundError('Notebook-09-Ergebnisse nicht gefunden. Bitte Notebook 09 vollständig ausführen.')
if RECON_CORE_PATH is None:
    raise FileNotFoundError('19_case_level_reconciliation_core.csv nicht gefunden. Bitte Notebook 08 vollständig ausführen.')
TABLE_SOURCE = PRED_ROOT / 'tables' if (PRED_ROOT / 'tables').exists() else PRED_ROOT
FIGURE_SOURCE = PRED_ROOT / 'figures' if (PRED_ROOT / 'figures').exists() else PRED_ROOT
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'final_prediction_quality_closure_thesis_assets'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for d in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Prediction source:', PRED_ROOT)
print('Reconciliation core:', RECON_CORE_PATH)
print('Output root:', OUTPUT_ROOT)


In [ ]:
# Hilfsfunktionen
created_tables, created_figures = ([], [])

def find_table(filename):
    candidates = [TABLE_SOURCE / filename, PRED_ROOT / filename]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(filename)
    return path

def save_csv(df, filename, index=False):
    path = TABLE_DIR / filename
    df.to_csv(path, index=index, encoding='utf-8-sig')
    created_tables.append(path)
    return path

def save_json(obj, filename):
    path = TABLE_DIR / filename
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    created_tables.append(path)
    return path

def save_fig(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches='tight')
    plt.close(fig)
    created_figures.append(path)
    return path

def robust_to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return series.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'])
NAME_MAP = {'label_scd_p90_or_global': 'Original SCD', 'label_scd_no_inspection_p90_or': 'Residual SCD', 'label_reopened_official': 'Reopened', 'label_late_payment_official_stable': 'Late Payment strict'}


In [ ]:
# Modellergebnisse laden
selected = pd.read_csv(find_table('13_selected_models_test_results.csv'))
bootstrap = pd.read_csv(find_table('14_selected_models_bootstrap_ci.csv'))
validation_selected = pd.read_csv(find_table('12_validation_selected_models.csv'))
evaluation = pd.read_csv(find_table('06_all_model_evaluation.csv'))
group_errors = pd.read_csv(find_table('19_selected_models_group_error_analysis.csv'))
permutation = pd.read_csv(find_table('16_selected_models_permutation_importance.csv'))
calibration = pd.read_csv(find_table('15_selected_models_calibration.csv'))
sensitivity2017 = pd.read_csv(find_table('20_selected_models_2017_sensitivity.csv'))
comparison_90_120 = pd.read_csv(find_table('17_time90_vs_time120_comparison.csv'))
ablation = pd.read_csv(find_table('18_target_specific_ablation_effects.csv'))
core = pd.read_csv(RECON_CORE_PATH, dtype={'case:concept:name': str})
print('Selected models:', selected.shape)
print('Core:', core.shape)
display(selected)


In [ ]:
# Vorhersagen prüfen
prediction_path = find_table('07_validation_test_predictions.csv')
parts = []
for chunk in pd.read_csv(prediction_path, usecols=['case:concept:name', 'split'], dtype={'case:concept:name': str, 'split': str}, chunksize=400000):
    parts.append(chunk.drop_duplicates())
case_splits = pd.concat(parts, ignore_index=True).drop_duplicates('case:concept:name')
core = core.merge(case_splits, on='case:concept:name', how='left')
core.loc[(core['case_year'].astype(str) == '2015') & core['split'].isna(), 'split'] = 'train'
split_check = core['split'].value_counts(dropna=False).rename_axis('split').reset_index(name='n_cases')
save_csv(split_check, '01_reproduced_split_counts.csv')
display(split_check)


In [ ]:
# Zielvariablen prüfen
for col in ['label_scd_p90_or_global', 'label_scd_no_inspection_p90_or', 'label_reopened_official', 'label_late_payment_official_stable']:
    if col in core.columns:
        core[col] = robust_to_bool(core[col])
configs = [('Original SCD', 'label_scd_p90_or_global', 'event_count', 'combined_rework_extra'), ('Residual SCD', 'label_scd_no_inspection_p90_or', 'event_count_no_inspection', 'rework_no_inspection')]
threshold_rows, agreement_rows = ([], [])
for label_name, original_col, metric_a, metric_b in configs:
    global_a = float(core[metric_a].quantile(0.9))
    global_b = float(core[metric_b].quantile(0.9))
    train_a = float(core.loc[core['split'].eq('train'), metric_a].quantile(0.9))
    train_b = float(core.loc[core['split'].eq('train'), metric_b].quantile(0.9))
    train_label_col = original_col + '_train_threshold'
    core[train_label_col] = (core[metric_a] >= train_a) | (core[metric_b] >= train_b)
    threshold_rows.append({'label': label_name, 'metric_a': metric_a, 'global_p90_a': global_a, 'train_p90_a': train_a, 'metric_b': metric_b, 'global_p90_b': global_b, 'train_p90_b': train_b})
    for split, group in core.groupby('split'):
        intersection = int((group[original_col] & group[train_label_col]).sum())
        union = int((group[original_col] | group[train_label_col]).sum())
        agreement_rows.append({'label': label_name, 'split': split, 'n_cases': len(group), 'global_positive_pct': group[original_col].mean() * 100, 'train_threshold_positive_pct': group[train_label_col].mean() * 100, 'agreement_pct': (group[original_col] == group[train_label_col]).mean() * 100, 'jaccard': intersection / union if union else np.nan, 'different_cases': int((group[original_col] != group[train_label_col]).sum())})
threshold_df = pd.DataFrame(threshold_rows)
agreement_df = pd.DataFrame(agreement_rows)
save_csv(threshold_df, '02_target_thresholds_global_vs_train.csv')
save_csv(agreement_df, '03_target_threshold_provenance_agreement.csv')
display(threshold_df)
display(agreement_df)


In [ ]:
# Auswertbarkeit der Zielvariablen
eligibility_rows = []
for cutoff in [30, 60, 90, 120]:
    for split, group in core.groupby('split'):
        completed = group['duration_days'] <= cutoff
        eligibility_rows.append({'prefix': f'time{cutoff}d', 'split': split, 'n_cases': len(group), 'completed_by_cutoff': int(completed.sum()), 'completed_pct': completed.mean() * 100, 'active_at_cutoff': int((~completed).sum()), 'active_pct': (~completed).mean() * 100, 'min_duration_days': group['duration_days'].min(), 'median_duration_days': group['duration_days'].median()})
for split, group in core.groupby('split'):
    completed = group['event_count'] <= 20
    eligibility_rows.append({'prefix': 'event20', 'split': split, 'n_cases': len(group), 'completed_by_cutoff': int(completed.sum()), 'completed_pct': completed.mean() * 100, 'active_at_cutoff': int((~completed).sum()), 'active_pct': (~completed).mean() * 100, 'min_duration_days': group['duration_days'].min(), 'median_duration_days': group['duration_days'].median()})
eligibility_df = pd.DataFrame(eligibility_rows)
save_csv(eligibility_df, '04_prefix_operational_eligibility.csv')
display(eligibility_df)


In [ ]:
# Finale Modellergebnisse
pr_ci = bootstrap[bootstrap['metric'].eq('pr_auc')][['target', 'estimate', 'ci_lower_95', 'ci_upper_95']]
pr_ci = pr_ci.rename(columns={'estimate': 'pr_auc_ci_estimate'})
final = selected.merge(pr_ci, on='target', how='left')
final['target_name'] = final['target'].map(NAME_MAP)
final['baseline_pr_auc'] = final['prevalence_pct'] / 100
final['pr_auc_absolute_gain'] = final['pr_auc_average_precision'] - final['baseline_pr_auc']
final['pr_auc_lift_vs_baseline'] = final['pr_auc_average_precision'] / final['baseline_pr_auc']
role_map = {
    'label_scd_p90_or_global': 'Primäres Prediction-Ergebnis',
    'label_scd_no_inspection_p90_or': 'Robustheits- und Negativbefund',
    'label_reopened_official': 'Externer Benchmark und negativer Transportabilitätsbefund',
    'label_late_payment_official_stable': 'Zu selten; nur Sensitivität',
}
final['final_role'] = final['target'].map(role_map)
save_csv(final, '05_final_selected_models_with_baseline.csv')
display(final[['target_name', 'final_role', 'prefix_id', 'model', 'prevalence_pct', 'pr_auc_average_precision', 'baseline_pr_auc', 'pr_auc_lift_vs_baseline', 'roc_auc', 'precision', 'recall', 'f1', 'ci_lower_95', 'ci_upper_95']])


In [ ]:
# Validierung und Test vergleichen
validation_view = validation_selected[['target', 'prefix_id', 'scenario', 'model', 'pr_auc_average_precision', 'roc_auc', 'f1', 'prevalence_pct']].rename(columns={'pr_auc_average_precision': 'validation_pr_auc', 'roc_auc': 'validation_roc_auc', 'f1': 'validation_f1', 'prevalence_pct': 'validation_prevalence_pct'})
test_view = selected[['target', 'prefix_id', 'scenario', 'model', 'pr_auc_average_precision', 'roc_auc', 'f1', 'prevalence_pct']].rename(columns={'pr_auc_average_precision': 'test_pr_auc', 'roc_auc': 'test_roc_auc', 'f1': 'test_f1', 'prevalence_pct': 'test_prevalence_pct'})
generalization = validation_view.merge(test_view, on=['target', 'prefix_id', 'scenario', 'model'])
generalization['target_name'] = generalization['target'].map(NAME_MAP)
generalization['delta_test_minus_validation_pr_auc'] = generalization['test_pr_auc'] - generalization['validation_pr_auc']
generalization['delta_test_minus_validation_roc_auc'] = generalization['test_roc_auc'] - generalization['validation_roc_auc']
save_csv(generalization, '06_validation_test_generalization.csv')
display(generalization)


In [ ]:
# Fehler und Merkmalsanalyse
inspection_errors = group_errors[group_errors['split'].eq('test') & group_errors['group_variable'].eq('has_inspection_event_context')].copy()
inspection_errors['target_name'] = inspection_errors['target'].map(NAME_MAP)
save_csv(inspection_errors, '07_inspection_subgroup_error_summary.csv')
display(inspection_errors)

def feature_category(feature):
    text = str(feature).lower()
    if 'prefix_event_count' in text or 'rework' in text:
        return 'Frühe Akkumulation einer targetnahen Strukturkomponente'
    if 'inspection' in text or 'on_site' in text or 'onsite' in text:
        return 'Inspection-Kontext'
    if 'department' in text:
        return 'Organisatorischer Kontext'
    if 'resource' in text:
        return 'Ressourcenkontext'
    if 'duration' in text:
        return 'Bis zum Prefix beobachtete Zeit'
    if any((token in text for token in ['activity', 'subprocess', 'doctype', 'combined'])):
        return 'Prozesskontext'
    return 'Sonstiges frühes Merkmal'
feature_rows = []
for target, group in permutation.groupby('target'):
    top = group.sort_values('importance_mean', ascending=False).head(20).copy()
    top['target_name'] = target and NAME_MAP.get(target, target)
    top['interpretation_category'] = top['raw_feature'].map(feature_category)
    top['interpretation_warning'] = 'Assoziation im ausgewählten Modell; keine Kausalwirkung.'
    feature_rows.append(top)
features = pd.concat(feature_rows, ignore_index=True)
save_csv(features, '08_top_permutation_features_interpreted.csv')
display(features.groupby('target_name').head(10))


In [ ]:
# PR AUC und Baseline abbilden
plot_final = final.sort_values('pr_auc_average_precision', ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 5.2))
x = np.arange(len(plot_final))
y = plot_final['pr_auc_average_precision'].to_numpy()
low = plot_final['ci_lower_95'].to_numpy()
high = plot_final['ci_upper_95'].to_numpy()
ax.bar(x, y, label='Test PR-AUC')
ax.errorbar(x, y, yerr=[y - low, high - y], fmt='none', ecolor='black', capsize=4)
ax.scatter(x, plot_final['baseline_pr_auc'], marker='D', label='Prävalenzbaseline')
ax.set_xticks(x)
ax.set_xticklabels(plot_final['target_name'], rotation=18, ha='right')
ax.set_ylabel('PR-AUC')
ax.set_title('Validation-ausgewählte Modelle auf Test 2016')
ax.grid(axis='y', alpha=0.25)
ax.legend()
save_fig(fig, 'fig_01_test_pr_auc_vs_baseline.png')
fig, ax = plt.subplots(figsize=(9, 5.2))
g = generalization.sort_values('test_pr_auc', ascending=False).reset_index(drop=True)
x = np.arange(len(g))
width = 0.35
ax.bar(x - width / 2, g['validation_pr_auc'], width, label='Validation 2015')
ax.bar(x + width / 2, g['test_pr_auc'], width, label='Test 2016')
ax.scatter(x, g['test_prevalence_pct'] / 100, marker='D', label='Test-Prävalenz')
ax.set_xticks(x)
ax.set_xticklabels(g['target_name'], rotation=18, ha='right')
ax.set_ylabel('PR-AUC')
ax.set_title('Zeitliche Transportabilität')
ax.grid(axis='y', alpha=0.25)
ax.legend()
save_fig(fig, 'fig_02_validation_vs_test.png')
sub = evaluation[evaluation['target'].eq('label_scd_p90_or_global') & evaluation['scenario'].eq('observed_prefix') & evaluation['threshold_policy'].eq('fixed_0_5') & evaluation['model'].isin(['logreg_balanced', 'rf_balanced']) & evaluation['split'].isin(['validation', 'test'])].copy()
order = ['static_first_event', 'event20', 'time30d', 'time60d', 'time90d', 'time120d']
labels = ['First event', '20 events', '30 d', '60 d', '90 d', '120 d']
fig, ax = plt.subplots(figsize=(9.4, 5.2))
for (model, split), group in sub.groupby(['model', 'split']):
    values = group.set_index('prefix_id').reindex(order)
    ax.plot(range(len(order)), values['pr_auc_average_precision'], marker='o', label=f'{model} | {split}')
ax.set_xticks(range(len(order)))
ax.set_xticklabels(labels)
ax.set_ylabel('PR-AUC')
ax.set_title('Original SCD über den beobachteten Prefix')
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
save_fig(fig, 'fig_03_original_scd_prefix_curve.png')
plot_group = inspection_errors[inspection_errors['target'].isin(['label_scd_p90_or_global', 'label_scd_no_inspection_p90_or', 'label_reopened_official'])].copy()
plot_group['label'] = plot_group['target_name'] + ' | Inspection=' + plot_group['group_value'].astype(str)
fig, ax = plt.subplots(figsize=(10, 5.3))
x = np.arange(len(plot_group))
ax.bar(x, plot_group['recall'])
ax.set_xticks(x)
ax.set_xticklabels(plot_group['label'], rotation=28, ha='right')
ax.set_ylabel('Recall')
ax.set_ylim(0, 1.05)
ax.set_title('Erkennungsrate nach Inspection-Kontext')
ax.grid(axis='y', alpha=0.25)
save_fig(fig, 'fig_04_inspection_group_recall.png')
plot_agreement = agreement_df[agreement_df['split'].eq('test')].set_index('label').reindex(['Original SCD', 'Residual SCD']).reset_index()
fig, ax = plt.subplots(figsize=(7.5, 4.8))
x = np.arange(len(plot_agreement))
width = 0.35
ax.bar(x - width / 2, plot_agreement['global_positive_pct'], width, label='Globales Label')
ax.bar(x + width / 2, plot_agreement['train_threshold_positive_pct'], width, label='Trainingsbasierte Schwelle')
ax.set_xticks(x)
ax.set_xticklabels(plot_agreement['label'])
ax.set_ylabel('Positivanteil Test (%)')
ax.set_title('Target-Schwellenprovenienz')
ax.grid(axis='y', alpha=0.25)
ax.legend()
save_fig(fig, 'fig_05_threshold_provenance.png')
print('Figures created:', len(created_figures))
